# Paper 2 · Cosmos-Predict2 on Colab

Runs the official [cosmos-predict2](https://github.com/nvidia-cosmos/cosmos-predict2) repo (commit `661da477`). Only the LoRA config, data subset script and metric code are my own (`eyecode-official/`).

**Experiment:** LoRA post-train Cosmos-Predict2-2B-Video2World (480p, 16 fps) on a small subset of GR1-100, then compare with the base model on held-out episodes. Input = first frame + prompt, metrics = PSNR / SSIM vs the real video (conditioning frame excluded).

**Needs:** A100-class GPU (>=32 GB), HF token, access to `nvidia/Cosmos-Predict2-2B-Video2World` and `meta-llama/Llama-Guard-3-8B`. T4 / L4 won't fit.

## 0 · Hardware check

In [ ]:
# check gpu (needs >=32GB, ampere+)
import torch, subprocess
# need gpu runtime
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (A100 recommended)"
# NGPU used later for torchrun
NGPU = torch.cuda.device_count()
props = [torch.cuda.get_device_properties(i) for i in range(NGPU)]
for i, p in enumerate(props):
    print(f"GPU{i}: {p.name}  {p.total_memory/2**30:.0f} GiB  sm_{p.major}{p.minor}")
# min vram (GiB)
MIN_GIB = 32
ok = all(p.total_memory / 2**30 >= MIN_GIB and p.major >= 8 for p in props)
print(f"NGPU={NGPU} -> launching with torchrun --nproc_per_node={NGPU}  ({'single process' if NGPU == 1 else 'DDP / multi-process'})")
# only warns, doesn't stop
if not ok:
    print(f"\n!! {MIN_GIB}+ GiB and an Ampere-or-newer GPU are required by the official code (cosmos-predict2 docs/performance.md). "
          "Switch to an A100 (Colab Pro) runtime; continuing will most likely run out of memory.")

GPU0: Tesla T4  15 GiB  sm_75
NGPU=1 -> launching with torchrun --nproc_per_node=1  (single process)

!! 32+ GiB and an Ampere-or-newer GPU are required by the official code (cosmos-predict2 docs/performance.md). Switch to an A100 (Colab Pro) runtime; continuing will most likely run out of memory.


In [ ]:
# setup eyecode-official helper code
import os, subprocess, sys
GLUE_REPO = "https://github.com/<your-user>/eyecode-official"  # <-- set to YOUR fork of this repo, or leave and upload the zip
if not os.path.isdir("/content/eyecode-official"):
    if "<your-user>" not in GLUE_REPO:
        subprocess.check_call(["git", "clone", "--depth", "1", GLUE_REPO, "/content/eyecode-official"])
    else:
        # no github user set -> upload the zip instead
        from google.colab import files
        print("Upload eyecode-official.zip"); up = files.upload()
        subprocess.check_call(["unzip", "-q", "-o", next(iter(up)), "-d", "/content"])
# extra libs for eval
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image", "opencv-python-headless", "pandas", "pyarrow", "matplotlib"])
sys.path.insert(0, "/content/eyecode-official")
# so %%bash cells can use $GLUE
os.environ["GLUE"] = "/content/eyecode-official"

Upload eyecode-official.zip


Saving eyecode-official.zip to eyecode-official.zip


## 1 · Repo + environment

In [ ]:
%%bash
# install official repo env (takes a few min)
# stop on any error
set -e
cd /content
# [ -d cosmos-predict2 ] || git clone https://github.com/nvidia-cosmos/cosmos-predict2.git
# pin to fixed commit
cd cosmos-predict2 && git checkout 661da4774b0ca41d082a0ecbeb47550bcf07e03f
pip install -q uv
uv sync --extra cu126  # creates .venv with the pinned torch 2.6 / transformer-engine stack (several minutes)
# sanity check env
.venv/bin/python scripts/test_environment.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 101.3 MB/s eta 0:00:00
Attempting to import critical packages...
[SUCCESS] torch found (v2.6.0+cu126)
[SUCCESS] torchvision found (v0.21.0+cu126)
[SUCCESS] diffusers found (v0.33.1)
[SUCCESS] transformers found (v4.51.3)
[SUCCESS] transformer_engine found (v1.13.0+e5edd6c)
[SUCCESS] transformer_engine.pytorch found
[SUCCESS] megatron.core found (v0.12.1)
[SUCCESS] flash_attn found (v2.6.3)
[SUCCESS] natten found (v0.21.0)
-----------------------------------------------------------
[SUCCESS] Cosmos-predict2 environment setup is successful!


Cloning into 'cosmos-predict2'...
Note: switching to '661da4774b0ca41d082a0ecbeb47550bcf07e03f'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 661da47 Update README.md
Using CPython 3.10.22
Creating virtual environment at: .venv
Resolved 162 packages in 1ms
Prepared 162 packages in 1m 23s
Installed 162 packages in 1.04s
 + accelerate==1.8.1
 + aniso8601==10.0.1
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + apex==0.1
 + asciitree==0.3.3
 + asttokens==3.0.0
 + attrs==25.1.0
 + av==15.0.0
 + bett

## 2 · HF login + checkpoints

In [ ]:
# check .venv was created
!ls ./cosmos-predict2/.venv/bin

accelerate		    ftfy		  pyftsubset
accelerate-config	    huggingface-cli	  pygmentize
accelerate-estimate-memory  imageio_download_bin  pygrun
accelerate-launch	    imageio_remove_bin	  py.test
accelerate-merge-weights    ipdb3		  pytest
activate		    ipython		  python
activate.bat		    ipython3		  python3
activate.csh		    isympy		  python3.10
activate.fish		    jp.py		  tabulate
activate.nu		    lsm2bin		  tiff2fsspec
activate.ps1		    markdown-it		  tiffcomment
activate_this.py	    modelscope		  tifffile
activate.xsh		    ninja		  tiny-agents
coverage		    nltk		  torchfrtrace
coverage3		    normalizer		  torchrun
coverage-3.10		    proton		  tqdm
deactivate.bat		    proton-viewer	  transformers-cli
diffusers-cli		    pulptest		  ttx
f2py			    pyav		  wandb
flask			    pydoc.bat		  wb
fonttools		    pyftmerge		  widsindex


In [ ]:
# get HF token (colab secrets, else ask)
import getpass, os
# try colab secrets first
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# fallback: paste it
except Exception:
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (read): ")

Hugging Face token (read): ··········


In [ ]:
%%bash
# download official checkpoints
set -e
cd /content/cosmos-predict2
uv pip install -q -U "huggingface_hub[cli]"
# 2B video2world, 480p, 16fps
.venv/bin/python scripts/download_checkpoints.py --model_types video2world --model_sizes 2B --fps 16 --resolution 480
du -sh checkpoints

Error downloading nvidia/Cosmos-Predict2-2B-Video2World: 403 Client Error. (Request ID: Root=1-6ad36881-62139588734b22f15f6658d5;f95a03f3-c9af-486d-a944-a1730d65a7d1)

Cannot access gated repo for url https://huggingface.co/nvidia/Cosmos-Predict2-2B-Video2World/resolve/f50c09f5d8ab133a90cac3f4886a6471e9ba3f18/model-480p-16fps.pt.
Access to model nvidia/Cosmos-Predict2-2B-Video2World is restricted and you are not in the authorized list. Visit https://huggingface.co/nvidia/Cosmos-Predict2-2B-Video2World to ask for access.
--------------------
Error downloading nvidia/Cosmos-Predict2-2B-Video2World: 403 Client Error. (Request ID: Root=1-6ad36882-7c1fea173b2df09d0e1a1cbd;378d1c00-55bf-44f2-94d9-e0c0e34c6038)

Cannot access gated repo for url https://huggingface.co/nvidia/Cosmos-Predict2-2B-Video2World/resolve/f50c09f5d8ab133a90cac3f4886a6471e9ba3f18/tokenizer/LICENSE.txt.
Access to model nvidia/Cosmos-Predict2-2B-Video2World is restricted and you are not in the authorized list. Visit https

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]/content/cosmos-predict2/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 45229.45 MB. The target location checkpoints/google-t5/t5-11b/.cache/huggingface/download only has 39307.21 MB free disk space.
  warnings.warn(
/content/cosmos-predict2/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 45229.45 MB. The target location checkpoints/google-t5/t5-11b only has 39307.21 MB free disk space.
  warnings.warn(
Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]


## 3 · Data: GR1-100 split + T5 embeddings
N_TRAIN videos for training, N_EVAL different ones held out (first frames extracted).

In [ ]:
# num train / held-out videos
N_TRAIN, N_EVAL = 24, 8  # small-scale budget; raise if you have time

In [ ]:
%%bash -s "$N_TRAIN" "$N_EVAL"
# get GR1-100, split train/eval, make T5 embeddings
# $1,$2 = N_TRAIN, N_EVAL
set -e
cd /content/cosmos-predict2
.venv/bin/huggingface-cli download nvidia/GR1-100 --repo-type dataset --local-dir datasets/benchmark_train/hf_gr1/
mkdir -p datasets/gr1_all/videos
mv datasets/benchmark_train/hf_gr1/gr1/*mp4 datasets/gr1_all/videos/
mv datasets/benchmark_train/hf_gr1/metadata.csv datasets/gr1_all/
# split + first frames of eval videos
python3 $GLUE/eyecode/prepare_subset.py --src datasets/gr1_all --out_train datasets/eyecode_gr1_train \
    --out_eval datasets/eyecode_gr1_eval --n_train $1 --n_eval $2
# official pre-processing script (writes metas/*.txt and t5_xxl/*.pickle)
# T5 embeddings for train set
.venv/bin/python -m scripts.get_t5_embeddings_from_groot_dataset \
    --dataset_path datasets/eyecode_gr1_train --meta_csv datasets/eyecode_gr1_train/metadata.csv
ls datasets/eyecode_gr1_train

Ignored error while writing commit hash to /root/.cache/huggingface/hub/datasets--nvidia--GR1-100/refs/main: [Errno 28] No space left on device: '/root/.cache/huggingface/hub'.
Fetching 95 files:   0%|          | 0/95 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/root/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/pathlib.py", line 1175, in mkdir
    self._accessor.mkdir(self, mode)
FileNotFoundError: [Errno 2] No such file or directory: 'datasets/benchmark_train/hf_gr1/.cache/huggingface'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/root/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/pathlib.py", line 1175, in mkdir
    self._accessor.mkdir(self, mode)
FileNotFoundError: [Errno 2] No such file or directory: 'datasets/benchmark_train/hf_gr1/.cache'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/r

CalledProcessError: Command 'b'set -e\ncd /content/cosmos-predict2\n.venv/bin/huggingface-cli download nvidia/GR1-100 --repo-type dataset --local-dir datasets/benchmark_train/hf_gr1/\nmkdir -p datasets/gr1_all/videos\nmv datasets/benchmark_train/hf_gr1/gr1/*mp4 datasets/gr1_all/videos/\nmv datasets/benchmark_train/hf_gr1/metadata.csv datasets/gr1_all/\npython3 $GLUE/eyecode/prepare_subset.py --src datasets/gr1_all --out_train datasets/eyecode_gr1_train \\\n    --out_eval datasets/eyecode_gr1_eval --n_train $1 --n_eval $2\n# official pre-processing script (writes metas/*.txt and t5_xxl/*.pickle)\n.venv/bin/python -m scripts.get_t5_embeddings_from_groot_dataset \\\n    --dataset_path datasets/eyecode_gr1_train --meta_csv datasets/eyecode_gr1_train/metadata.csv\nls datasets/eyecode_gr1_train\n'' returned non-zero exit status 1.

## 4 · Baseline inference (no fine-tuning)

In [ ]:
import json, pathlib, subprocess
ROOT = pathlib.Path("/content/cosmos-predict2")
# template: first frames + prompts of held-out videos
tmpl = json.loads((ROOT / "datasets/eyecode_gr1_eval/batch_input_template.json").read_text())

# make_batch: fill {run} (base/lora) in output path
def make_batch(run):
    items = [{**t, "output_video": t["output_video"].replace("{run}", run)} for t in tmpl]
    p = ROOT / f"datasets/eyecode_gr1_eval/batch_{run}.json"
    p.write_text(json.dumps(items, indent=2)); return p

# sh: run cmd in repo, stop if it fails
def sh(cmd):
    print(cmd); r = subprocess.run(cmd, shell=True, cwd=ROOT, executable="/bin/bash"); assert r.returncode == 0, f"failed: {r.returncode}"

# disable fused attn
ENV = "NVTE_FUSED_ATTN=0"
# baseline inference (no finetune)
sh(f"{ENV} .venv/bin/torchrun --nproc_per_node={NGPU} --master_port=12341 -m examples.video2world "
   f"--model_size 2B --resolution 480 --fps 16 --num_gpus {NGPU} --batch_input_json {make_batch('base')} "
   f"--disable_guardrail --disable_prompt_refiner")

## 5 · LoRA post-training
Rank 16, EMA off, batch 1/GPU. 1 GPU = single process, >1 = DDP.

In [ ]:
import os, shutil
# training steps (small budget)
MAX_ITER = 300
# copy lora config into repo
shutil.copy("/content/eyecode-official/overlay/eyecode_gr1_lora_small.py", ROOT / "cosmos_predict2/configs/base/experiment/")
# ddp for multi-gpu, else fsdp
par = "ddp" if NGPU > 1 else "fsdp"  # fsdp + shard_size=0 on one GPU == no sharding
env = f"NVTE_FUSED_ATTN=0 EYECODE_MAX_ITER={MAX_ITER} EYECODE_SAVE_ITER=100 EYECODE_DATASET_DIR=datasets/eyecode_gr1_train"
# train lora, log to train.log
sh(f"{env} .venv/bin/torchrun --nproc_per_node={NGPU} --master_port=12341 -m scripts.train "
   f"--config=cosmos_predict2/configs/base/config.py -- experiment=predict2_video2world_lora_eyecode_gr1_small "
   f"model.config.train_architecture=lora trainer.distributed_parallelism={par} model.config.fsdp_shard_size=0 "
   f"2>&1 | tee train.log")

In [ ]:
import re
log = open(ROOT / "train.log").read()
# parse (iter, loss) from log
losses = [(int(i), float(l)) for i, l in re.findall(r"[Ii]teration[^\d]*(\d+).*?[Ll]oss[^\d\-]*([\d.]+)", log)]
# quick check
print(f"parsed {len(losses)} loss lines;", losses[:3], "...", losses[-3:])

## 6 · Inference with the LoRA checkpoint

In [ ]:
import glob
# find lora ckpts
ckpts = sorted(glob.glob(str(ROOT / "checkpoints/posttraining/video2world_lora/2b_eyecode_gr1_small/checkpoints/model/*.pt")))
print(ckpts); assert ckpts, "no LoRA checkpoint found - see train.log"
# inference with latest ckpt
sh(f"NVTE_FUSED_ATTN=0 .venv/bin/torchrun --nproc_per_node={NGPU} --master_port=12341 -m examples.video2world_lora "
   f"--model_size 2B --resolution 480 --fps 16 --num_gpus {NGPU} --use_lora --dit_path {ckpts[-1]} "
   f"--batch_input_json {make_batch('lora')} --disable_guardrail --disable_prompt_refiner")

## 7 · Evaluation: generated vs ground truth (held-out)

In [ ]:
# eval modules
from eyecode import video_metrics as vm, report
RES = pathlib.Path("/content/eyecode-official/results"); RES.mkdir(exist_ok=True)
GT = ROOT / "datasets/eyecode_gr1_eval/videos"
# PSNR/SSIM vs ground truth for each run
for run in ("base", "lora"):
    vm.save_scores(vm.evaluate_run(GT, ROOT / f"output/eval/{run}"), RES / f"video_scores_{run}.json")
# base vs lora summary table
lines, summ = report.video_section(RES, ["base", "lora"])
print("\n".join(lines))

## 8 · Best / worst cases

In [ ]:
import numpy as np, matplotlib.pyplot as plt, IPython.display as ipd
base, lora = (vm.load_scores(RES / f"video_scores_{r}.json") for r in ("base", "lora"))
b = {s.name: s for s in base}; l = {s.name: s for s in lora}
# delta = lora PSNR - base PSNR
names = sorted(set(b) & set(l)); delta = {n: l[n].psnr - b[n].psnr for n in names}
order = sorted(names, key=delta.get)
# best / worst 2 videos
worst, best = order[:2], order[-2:][::-1]
print("largest LoRA gain :", [(n, round(delta[n], 2)) for n in best])
print("largest LoRA loss :", [(n, round(delta[n], 2)) for n in worst])
# error growth with time: PSNR per frame averaged over videos
# common #frames
T = min(len(b[n].psnr_by_frame) for n in names)
# avg PSNR per frame, drift = first 10 - last 10
for run, d in (("base", b), ("lora", l)):
    pf = np.mean([d[n].psnr_by_frame[:T] for n in names], axis=0)
    print(f"{run}: PSNR first 10 frames {pf[:10].mean():.2f} dB -> last 10 frames {pf[-10:].mean():.2f} dB (drift {pf[:10].mean()-pf[-10:].mean():.2f} dB)")
# grid: GT | base | LoRA (mid frame)
fig, ax = plt.subplots(len(best + worst), 3, figsize=(10, 3 * len(best + worst)))
for r, n in enumerate(best + worst):
    for c, (lab, p) in enumerate((("ground truth", GT / n), ("base", ROOT / f"output/eval/base/{n}"), ("LoRA", ROOT / f"output/eval/lora/{n}"))):
        v = vm.read_video(p); ax[r, c].imshow(v[len(v) // 2 if c == 0 else min(len(v), 93) // 2]); ax[r, c].axis("off")
        ax[r, c].set_title(f"{lab} | {n[:22]} | ΔPSNR {delta[n]:+.2f}" if c == 2 else lab, fontsize=7)
fig.tight_layout(); fig.savefig(RES / "qualitative_best_worst.png", dpi=150); plt.show()
# side-by-side for best + worst
for n in best[:1] + worst[:1]:
    vm.side_by_side([GT / n, ROOT / f"output/eval/base/{n}", ROOT / f"output/eval/lora/{n}"], ["ground truth", "base", "LoRA"], RES / f"compare_{n}")
    ipd.display(ipd.Video(str(RES / f"compare_{n}"), embed=True, width=900))

## 9 · Save outputs

In [ ]:
%%bash
# zip results (input for notebook 02)
cd /content/cosmos-predict2 && cp train.log /content/eyecode-official/results/ \
 && python3 $GLUE/eyecode/report.py --results $GLUE/results \
 && zip -qr /content/cosmos_outputs.zip output/eval datasets/eyecode_gr1_eval/split.json datasets/eyecode_gr1_eval/batch_input_template.json datasets/eyecode_gr1_eval/videos /content/eyecode-official/results
ls -lh /content/cosmos_outputs.zip